# 02. Mini Photometry Pipeline Walkthrough

This notebook provides a simplified, step-by-step walkthrough of the core concepts in the `prose2` photometry pipeline. We'll use mock data to illustrate processes like source detection, aperture photometry, and differential photometry, without requiring actual FITS files.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, aperture_photometry

# --- Configuration ---
IMAGE_SIZE = 100
NUM_STARS = 10
TARGET_STAR_IDX = 0 # Index of the target star in our mock data
APERTURE_RADIUS = 5 # pixels
ANNULUS_INNER = 8
ANNULUS_OUTER = 12

print("Starting Mini Photometry Walkthrough...")


## 1. Generate Mock Image Data

We'll create a synthetic astronomical image with several stars and some background noise to simulate raw data.

In [ ]:
def generate_mock_image(size, num_stars, target_idx):
    image = np.random.normal(100, 5, size=(size, size)) # Background noise
    star_positions = []
    star_fluxes = []
    
    for i in range(num_stars):
        x = np.random.uniform(10, size - 10)
        y = np.random.uniform(10, size - 10)
        flux = np.random.uniform(500, 5000) if i != target_idx else 8000 # Make target brighter
        sigma = np.random.uniform(1.5, 2.5)
        
        Y, X = np.ogrid[-y:size-y, -x:size-x]
        gaussian = flux * np.exp(-(X**2 + Y**2) / (2 * sigma**2))
        image += gaussian
        star_positions.append((x, y))
        star_fluxes.append(flux)
        
    return image, np.array(star_positions), np.array(star_fluxes)

mock_image, star_positions, star_fluxes = generate_mock_image(IMAGE_SIZE, NUM_STARS, TARGET_STAR_IDX)

plt.figure(figsize=(8, 8))
plt.imshow(mock_image, origin='lower', cmap='viridis')
plt.title("Mock Astronomical Image")
plt.colorbar(label="Counts")
plt.scatter(star_positions[:, 0], star_positions[:, 1], marker='x', color='red', s=100, label='True Star Positions')
plt.legend()
plt.show()


## 2. Source Detection

Identify stars in the image. This step is crucial for defining where photometry apertures will be placed.

In [ ]:
# Using DAOStarFinder (from photutils) to detect sources
# For simplicity, we'll assume a constant FWHM
fwhm = 3.0
daofind = DAOStarFinder(fwhm=fwhm, threshold=5.*np.median(mock_image))
sources = daofind(mock_image)

if sources is None:
    print("No sources found. Adjust threshold or FWHM.")
else:
    positions = np.transpose((sources['xcentroid'], sources['ycentroid']))
    
    plt.figure(figsize=(8, 8))
    plt.imshow(mock_image, origin='lower', cmap='viridis')
    plt.colorbar(label="Counts")
    plt.scatter(positions[:, 0], positions[:, 1], marker='o', facecolors='none', edgecolors='red', s=100, label='Detected Sources')
    plt.title("Source Detection")
    plt.legend()
    plt.show()
    
    print(f"Detected {len(sources)} sources:")
    for i, s in enumerate(sources):
        print(f"  Star {i}: X={s['xcentroid']:.2f}, Y={s['ycentroid']:.2f}, Flux={s['flux']:.2f}")


## 3. Aperture Photometry

Once sources are detected, we place circular apertures around them to measure their flux, and annuli for background estimation.

In [ ]:
if sources is not None:
    apertures = CircularAperture(positions, r=APERTURE_RADIUS)
    annuli = CircularAnnulus(positions, r_in=ANNULUS_INNER, r_out=ANNULUS_OUTER)
    
    # Perform photometry
    phot_table = aperture_photometry(mock_image, apertures)
    bkg_table = aperture_photometry(mock_image, annuli)
    
    # Estimate background and subtract
    bkg_mean = bkg_table['aperture_sum'] / annuli.area
    bkg_sum = bkg_mean * apertures.area
    final_flux = phot_table['aperture_sum'] - bkg_sum

    phot_table['background_flux'] = bkg_sum
    phot_table['final_flux'] = final_flux

    plt.figure(figsize=(8, 8))
    plt.imshow(mock_image, origin='lower', cmap='viridis')
    plt.colorbar(label="Counts")
    apertures.plot(color='red', lw=1.5, alpha=0.8, label='Apertures')
    annuli.plot(color='white', lw=1.5, alpha=0.8, label='Annuli')
    plt.title("Aperture Photometry")
    plt.legend()
    plt.show()

    print("Photometry Results (first 5 sources):")
    print(phot_table['id', 'xcenter', 'ycenter', 'aperture_sum', 'background_flux', 'final_flux'])
else:
    print("Skipping aperture photometry as no sources were detected.")


## 4. Differential Photometry (Simplified)

Differential photometry uses comparison stars to remove atmospheric and instrumental variations. Here, we'll simulate this by dividing the target's flux by a composite comparison flux.

In [ ]:
if sources is not None and len(sources) > 1:
    # Find the target star (assuming it's the brightest for simplicity in mock data)
    target_flux = phot_table['final_flux'][np.argmax(sources['flux'])]
    
    # Use all other stars as comparison stars (simple average for demo)
    comparison_indices = np.delete(np.arange(len(sources)), np.argmax(sources['flux']))
    comparison_fluxes = phot_table['final_flux'][comparison_indices]
    
    # Simulate some variation over time for comparison stars to show detrending effect
    time_variation = np.linspace(1, 1.05, len(comparison_fluxes)) # a subtle trend
    comparison_flux_composite = np.mean(comparison_fluxes * time_variation) # Simplified composite

    differential_flux = target_flux / comparison_flux_composite

    print(f"Target Flux: {target_flux:.2f}")
    print(f"Composite Comparison Flux: {comparison_flux_composite:.2f}")
    print(f"Differential Flux (relative to composite comparison): {differential_flux:.4f}")
    
    # In a real scenario, this would be done over many images to form a lightcurve
    # For demonstration, we'll just show a single frame's differential flux
    # (This section is conceptual as it typically requires a time series of images)
    plt.figure(figsize=(6, 4))
    plt.bar(['Target', 'Comparison Composite'], [target_flux, comparison_flux_composite], color=['skyblue', 'lightcoral'])
    plt.title("Conceptual Differential Photometry (Single Frame)")
    plt.ylabel("Flux")
    plt.show()
    
    print("\nThis demonstrates the principle. In a full pipeline, these steps are applied to hundreds or thousands of frames to produce a differential lightcurve over time.")
else:
    print("Skipping differential photometry as fewer than 2 sources were detected.")


## 5. Summary

This notebook walked through the fundamental steps of photometry: mock data generation, source detection, aperture photometry, and the conceptual basis of differential photometry. The actual `prose2` pipeline automates and refines these steps for robust exoplanet lightcurve extraction.